In [39]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [40]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDs


In [41]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

initialise our model here

In [42]:
MODEL = 'gpt-4.1-mini'

### The usage of tools

#### Tools allow us to pivot from a Chatbot (which just talks) to an AI Agent (which takes action).

This is the most critical part of Agentic AI. This pattern is often called the "Round Trip" or "Re-Act" (Reason + Act) loop.

Unlike a normal chat where you send text and get text, here is the flow we need to implement in code:

First Call: Send user text → LLM decides to call a function.

Execution: Your Python code sees the request, runs the function, and captures the output.

Second Call: You send the Function Output back to the LLM.

Final Answer: LLM reads the output and answers the user in plain English.

Here is the complete, runnable implementation.

##### 1. Setup: Define the "Real" Functions
##### First, we need the actual Python functions and a "Dispatch Table" (a dictionary) so the AI can map the string name "get_leave_balance" to the actual function code.

## Leave Agent Tools

In [43]:
import json
import random

# --- MOCK DATABASE ---
db = {
    "mark_tan": {"annual": 14, "medical": 14, "family": 3},
    "jane_doe": {"annual": 2, "medical": 10, "family": 0}
}


# --- ACTUAL PYTHON FUNCTIONS ---
def get_leave_balance(employee_id, leave_type):
    # Simulate DB lookup
    user_data = db.get(employee_id)
    if not user_data:
        return json.dumps({"error": "Employee not found"})
    
    balance = user_data.get(leave_type.lower())
    if balance is None:
        return json.dumps({"error": f"Invalid leave type: {leave_type}"})
        
    return json.dumps({"balance": balance, "unit": "days"})

def submit_leave_request(employee_id, leave_type, days):
    case_number = random.randint(10000,99999)

    # Simulate a "POST" request
    print(f"--- SYSTEM: Submitting {days} days of {leave_type} leave for {employee_id} ---")
    return json.dumps({"status": "success", "message": f"Request ID #MW{case_number} approved pending manager review."})

def draft_leave_request(employee_id, leave_type, start_date, end_date):
    draft = {
        "employee_id": employee_id,
        "leave_type": leave_type,
        "start_date": start_date,
        "end_date": end_date,
        "status": "Draft",
    }

    print(f"--- SYSTEM: Draft created for {employee_id} ({leave_type} leave) from {start_date} to {end_date} ---")
    return json.dumps({
        "status": "draft",
        "message": "Draft leave request created.",
        "draft": draft
    })


2. The Tool Definitions (Schema)
This is what we send to OpenAI to "teach" it about the tools above.

In [44]:
leave_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_leave_balance",
            "description": "Get the remaining leave days for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "leave_type": {"type": "string", "enum": ["annual", "medical", "family"]},
                },
                "required": ["employee_id", "leave_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_leave_request",
            "description": "Submit a leave application after user has reviewed draft",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "leave_type": {"type": "string"},
                    "days": {"type": "integer"},
                },
                "required": ["employee_id", "leave_type", "days"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "draft_leave_request",
            "description": "Create a draft leave request before submitting.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "leave_type": {"type": "string"},
                    "start_date": {"type": "string", "description": "YYYY-MM-DD"},
                    "end_date": {"type": "string", "description": "YYYY-MM-DD"}
                },
                "required": ["employee_id", "leave_type", "start_date", "end_date"]
            }
        }
    }
]

In [45]:
def agent_leaves():
    return {
        "system": """
            You are a helpful HR assistant for bank staff. You are tasked with helping staff with applying for leave.
            Before submitting a request always make sure you have drafted the request and presented to the user. Include request ID when appropriate.
            Whenever you call a tool, always include the output in the response.
            Current Employee ID: mark_tan.
        """,
        "tools": leave_tools
    }

## Devices Agent 

In [46]:
device_db = [
    {"id": 1, "name": "2M HDMI Cable", "cost": 6.50},
    {"id": 2, "name": "Wireless Mouse", "cost": 15.00},
    {"id": 3, "name": "Mechanical Keyboard", "cost": 45.00},
    {"id": 4, "name": "27-inch Monitor", "cost": 230.00},
    {"id": 5, "name": "USB-C Hub", "cost": 25.50},
    {"id": 6, "name": "External Hard Drive 1TB", "cost": 65.00},
    {"id": 7, "name": "Laptop Stand", "cost": 30.00},
    {"id": 8, "name": "Webcam 1080p", "cost": 40.00}
]

def get_available_devices():
    return json.dumps(device_db)


def submit_device_request(employee_id, device_id, device_name):
    case_number = random.randint(10000,99999)

    print(f"--- SYSTEM: Submitting request for {device_id}: {device_name} for {employee_id} ---")
    return json.dumps({"status": "success", "message": f"Request ID #MW{case_number} submitted pending manager review."})

def draft_device_request(employee_id, device_id, device_name):
    draft = {
        "employee_id": employee_id,
        "device_id": device_id,
        "device_name": device_name,
        "status": "Draft",
    }

    print(f"--- SYSTEM: Draft created for {employee_id} (device request) for {device_name} ---")
    return json.dumps({
        "status": "draft",
        "message": "Draft device request created.",
        "draft": draft
    })

In [47]:
device_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_available_devices",
            "description": "Return a list of all available devices that employees can request.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_device_request",
            "description": "Submit a device request for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "device_id": {"type": "integer"},
                    "device_name": {"type": "string"},
                },
                "required": ["employee_id", "device_id", "device_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "draft_device_request",
            "description": "Create a draft device request for an employee before final submission.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {"type": "string"},
                    "device_id": {"type": "integer"},
                    "device_name": {"type": "string"},
                },
                "required": ["employee_id", "device_id", "device_name"],
            },
        },
    }
]

In [48]:
def agent_devices():
    return {
        "system": """
            You are a helpful HR assistant for bank staff. You are tasked with helping staff with requesting for new devices.
            Before submitting a request always make sure you have drafted the request and presented to the user. Include request ID when appropriate.
            Whenever you call a tool, always include the output in the response.
            Current Employee ID: mark_tan.
        """,
        "tools": device_tools
    }

In [49]:
feature_db = [
    {
        "id": 1,
        "feature": "Leaves Agent",
        "description": "This agent helps you to apply for leaves"
    },
    {
        "id": 2,
        "feature": "Devices Agent",
        "description": "This agent helps you to apply for devices"
    }
]

def get_available_agents():
    return json.dumps(feature_db)

In [50]:
common_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_available_agents",
            "description": "Return a list of all available agents that can assist with user requests & workflows",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            },
        },
    },
]

In [51]:
def agent_common():
    return {
        "system": """
            You are a helpful HR assistant for bank staff. You are tasked with helping staff with general requests
            and listing workflows you can support them with
            Current Employee ID: mark_tan.
        """,
        "tools": common_tools
    }

In [52]:
def pick_agent(user_message):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a routing classifier for feature requests."
                           "Decide which agent should answer: 'leave', 'devices' or 'general."

                           "Leave: Involves querying leave balance, drafting and submitting leave request"
                           "Devices: Involves requesting for new device accessories such as cables and peripherals"
                           "General: Other unrelated queries"
            },
            {"role": "user", "content": user_message}
        ]
    )

    intent = response.choices[0].message.content.lower()

    if "leave" in intent:
        return "leave"
    if "devices" in intent:
        return "devices"
    return "general"


In [53]:
# --- DISPATCH TABLE (Mapping String -> Function) ---
available_functions = {
    "get_leave_balance": get_leave_balance,
    "submit_leave_request": submit_leave_request,
    "draft_leave_request": draft_leave_request,
    "get_available_devices": get_available_devices,
    "submit_device_request": submit_device_request,
    "draft_device_request": draft_device_request,
    "get_available_agents": get_available_agents
}

3. The Agent Logic (The Loop)
I have removed stream=True for this example because handling streaming while doing function calling creates very complex code. It is safer to start with standard generation.

In [54]:
def run_agent(config, message, history):
    # 1. Format history for OpenAI (cleaning up Gradio stuff)
    history_formatted = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # 2. Add System Prompt
    system_prompt = config["system"]
    messages = [{"role": "system", "content": system_prompt}] + history_formatted + [{"role": "user", "content": message}]

    # --- FIRST API CALL (The "Thinking" Phase) ---
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=config["tools"],
        tool_choice="auto", 
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # --- CHECK: Did the AI want to run a tool? ---
    if tool_calls:
        # A. Add the AI's "thought" (request to call tool) to the history
        messages.append(response_message) 
        print(tool_calls)
        
        # B. Iterate over requested tools (AI might call 2 at once!)
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            # C. Execute the actual Python function
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)
            
            # D. Append the RESULT to the history
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )
            
        # --- SECOND API CALL (The "Answering" Phase) ---
        # Now the AI sees the tool output and generates the final text
        second_response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
        )
        return second_response.choices[0].message.content

    else:
        # If no tool was needed, just return the text
        return response_message.content

In [55]:
def unified_chat(message, history):
    agent_name = pick_agent(message)

    if agent_name == "leave":
        config = agent_leaves()
    elif agent_name == "devices":
        config = agent_devices()
    else:
        config = agent_common()

    return run_agent(config, message, history)

In [ ]:
import gradio as gr

custom_textbox = gr.Textbox(
    label="Your message:", 
    info="Enter your access request", # Updated info text
    lines=7,
    submit_btn=True
)

view = gr.ChatInterface(
    fn=unified_chat,
    title="SimpliAsk", 
    textbox=custom_textbox,
    submit_btn="Submit Request",
    examples=[
        "I would like to apply for 7 days of annual leave.",
        "I would like to request for HDMI Cable",
        "List workflows you can assist me with"
    ],
    flagging_mode="never"
)

view.launch()

/tmp/ipykernel_148670/3147030537.py:10: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  view = gr.ChatInterface(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


[ChatCompletionMessageFunctionToolCall(id='call_PiA7fiw5LzGkZy8XrcBBGZUW', function=Function(arguments='{}', name='get_available_agents'), type='function')]
